# AGORA NER Pipeline — Demos

Five self-contained demos showing why the LLM-based, community-aware pipeline produces entities that a general-purpose NER tool cannot.

Each demo runs locally from existing pipeline outputs — no API calls, no reprocessing.

| # | Demo | Key point |
|---|---|---|
| 1 | Disambiguation in action | Same word → 41 distinct officials |
| 2 | Text-in / entities-out | Auditable extraction with context fields |
| 3 | The "Secretary" problem | Surface-form collapse vs. structured resolution |
| 4 | Party asymmetry on entity types | Political signal only visible via role/legislation entities |
| 5 | Entity graph neighborhood | Relational structure, not just entity lists |

In [1]:
import json, os, collections
from pathlib import Path

ROOT = Path("..").resolve()
MEMORY_DIR  = ROOT / "pipeline" / "agents" / "memory"
ENTITIES    = ROOT / "pipeline" / "agents" / "output" / "entities_canonicalized.jsonl"
FULLTEXT    = ROOT / "data" / "fulltext"
COSPONSOR   = ROOT / "knowledge_graph" / "graph_data" / "agora_cosponsors_long.csv"
SPONSORS    = ROOT / "knowledge_graph" / "graph_data" / "agora_with_sponsors.csv"
GRAPH       = ROOT / "pipeline" / "multiplex_graph" / "layer_3_entity.graphml"

def load_entities():
    recs = {}
    with open(ENTITIES) as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                recs[str(r["agora_id"])] = r
    return recs

print("Paths OK. Ready.")

Paths OK. Ready.


---
## Demo 1 — Disambiguation in action

**Setup:** US legislation uses short tokens ("Secretary", "Director", "the Department") that change meaning from bill to bill. The pipeline maintains per-community memory files that resolve each token to the correct named referent, inferred from the bill's definitions section.

**The question:** How many distinct officials does "secretary" resolve to across the corpus?

In [2]:
WORD = "secretary"   # try also: "director", "administrator", "commission", "department"

resolutions = []  # (community_id, resolution_text)
for fname in sorted(os.listdir(MEMORY_DIR)):
    if not fname.endswith("_memory.json"):
        continue
    d = json.load(open(MEMORY_DIR / fname))
    rules = d.get("disambiguation_rules", {})
    if WORD in rules:
        resolutions.append((d["community_id"], rules[WORD]))

print(f'"{WORD}" is disambiguated in {len(resolutions)} communities\n')
print(f"{'Community':<20}  Resolution")
print("-" * 80)
for cid, val in resolutions:
    print(f"  {cid:<18}  {val[:75]}")

"secretary" is disambiguated in 41 communities

Community             Resolution
--------------------------------------------------------------------------------
  community:001       In SEC. 7227(b), 'Secretary' refers to Secretary of Homeland Security per c
  community:002       Secretary of Defense per (f)(6) explicit definition within this section
  community:003       Explicitly defined as Secretary of Energy in SEC. 2(3)
  community:005       Resolved to Secretary of Labor per section 13 of this document
  community:009       In SEC. 931(b)(1), 'the Secretary' refers to the Secretary of State based o
  community:010       Secretary of Agriculture (defined in SEC. 2(6))
  community:012       Refers to Secretary of Agriculture based on Farm, Food, and National Securi
  community:013       Resolved to Secretary of the department in which the Coast Guard is operati
  community:014       Resolved to Secretary of Energy in Section 3(a) consultation clause
  community:017       Refers t

In [3]:
# Which cabinet departments show up in those resolutions?
dept_hits = collections.Counter()
keywords = [
    "Defense", "Energy", "Commerce", "Agriculture", "Labor",
    "Education", "State", "Treasury", "Interior", "Transportation",
    "Health", "Homeland", "Housing", "Veterans"
]
for _, val in resolutions:
    for kw in keywords:
        if kw.lower() in val.lower():
            dept_hits[kw] += 1
            break
    else:
        dept_hits["Other"] += 1

print("Resolved departments:")
for dept, cnt in dept_hits.most_common():
    bar = "█" * cnt
    print(f"  {dept:<15} {bar} {cnt}")

Resolved departments:
  Energy          ████████ 8
  Agriculture     ██████ 6
  Health          ██████ 6
  Defense         ████ 4
  Commerce        ████ 4
  Homeland        ███ 3
  Labor           ██ 2
  Education       ██ 2
  Transportation  ██ 2
  Veterans        ██ 2
  Other           █ 1
  Treasury        █ 1


---
## Demo 2 — Text-in / entities-out with auditable context

**Setup:** Every extracted entity carries a `context` field — the exact substring from the source text that triggered the extraction. This makes every entity auditable: you can trace it back to the clause that defined it.

**The question:** What does doc 10 (intelligence community AI reporting requirements) look like before and after extraction?

In [4]:
DOC_ID = "10"   # try any agora_id that has a fulltext file

# Raw text (first 600 chars)
text = (FULLTEXT / f"{DOC_ID}.txt").read_text(errors="ignore")
print("── RAW TEXT (first 600 chars) " + "─" * 45)
print(text[:600])
print()

── RAW TEXT (first 600 chars) ─────────────────────────────────────────────
SEC. 6721. REPORTS ON INTEGRATION OF ARTIFICIAL INTELLIGENCE WITHIN INTELLIGENCE COMMUNITY.

(a) Reports by Elements of Intelligence Community.--Not later than 180 days after the date of the enactment of this Act, each senior official within an element of the intelligence community identified as a designated element lead pursuant to section 6702(b) shall submit to the congressional intelligence committees, the Subcommittee on Defense of the Committee on Appropriations of the Senate, and the Subcommittee on Defense of the Committee on Appropriations of the House of Representatives a report on t



In [5]:
recs = load_entities()
rec = recs[DOC_ID]

FIELDS = {
    "organizations": "name",
    "offices":       "name",
    "roles":         "title",
    "legislation_refs": "name",
    "named_docs":    "name",
}

print("── EXTRACTED ENTITIES " + "─" * 53)
for field, key in FIELDS.items():
    ents = rec.get(field, [])
    if not ents:
        continue
    print(f"\n{field.upper()} ({len(ents)})")
    for e in ents:
        name    = e.get(key, "")
        context = e.get("context", "")[:90]
        acronym = f" [{e['acronym']}]" if e.get("acronym") else ""
        print(f"  • {name}{acronym}")
        if context:
            print(f"      context: \"{context}\"")

── EXTRACTED ENTITIES ─────────────────────────────────────────────────────

ORGANIZATIONS (6)
  • intelligence community.
      context: "each senior official within an element of the intelligence community"
  • congressional intelligence committees
      context: "submit to the congressional intelligence committees"
  • Committee on Appropriations of the Senate
      context: "Subcommittee on Defense of the Committee on Appropriations of the Senate"
  • Committee on Appropriations of the House of Representatives
      context: "Subcommittee on Defense of the Committee on Appropriations of the House"
  • Office of the Director of National Intelligence [ODNI]
      context: "audit of the Office of the Director of National Intelligence"
  • United States Government
      context: "ensure the United States Government outpaces foreign adversaries in such field"

OFFICES (2)
  • Subcommittee on Defense of the Committee on Appropriations of the Senate
      context: "submit to...the Subcomm

In [6]:
# What disambiguation updates did the pipeline make while processing this doc?
updates = rec.get("disambiguation_updates", {})
if updates:
    print("── DISAMBIGUATION UPDATES APPLIED TO THIS DOC " + "─" * 28)
    for surface, resolved in updates.items():
        print(f"  \"{surface}\"  →  {resolved}")
else:
    print("No disambiguation updates logged for this doc (already resolved from community memory).")

print(f"\nModel: {rec.get('model')}  |  Chunks processed: {rec.get('chunks_processed')}  |  Tokens: {rec.get('prompt_tokens')}p / {rec.get('completion_tokens')}c")

── DISAMBIGUATION UPDATES APPLIED TO THIS DOC ────────────────────────────
  "Director"  →  Director of National Intelligence (per Office of the Director of National Intelligence context)
  "intelligence community"  →  Intelligence Community (standard organizational entity per existing extraction)

Model: claude-haiku-4-5-20251001  |  Chunks processed: 2  |  Tokens: 2586p / 1493c


---
## Demo 3 — The "Secretary" problem: surface collapse vs. structured resolution

**Setup:** A naïve NER tool returns the surface token "Secretary" 88 times across the corpus and puts all 88 mentions into one graph node. The pipeline resolves each occurrence to the correct cabinet official. This demo quantifies what information is destroyed by surface-form extraction.

**The question:** If you collapsed all "Secretary" mentions into one node, how many distinct policy domains would you be incorrectly merging?

In [7]:
import csv

# Load documents.csv for taxonomy tags
docs_tags = {}
TAG_PREFIXES = ("Applications:", "Harms:", "Strategies:", "Risk factors:", "Incentives:")
with open(ROOT / "data" / "documents.csv", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    tag_cols = [c for c in reader.fieldnames if c.startswith(TAG_PREFIXES)]
    for row in reader:
        aid = row.get("AGORA ID", "")
        active = [c for c in tag_cols if row.get(c, "").strip() not in ("", "0", "FALSE", "False", "No")]
        if active:
            docs_tags[aid] = active

# For each role entity named "Secretary" (bare), find which docs it appears in and their tags
bare_secretary_docs = set()
resolved_secretaries = collections.defaultdict(set)  # resolved_title -> set of aids

recs_all = load_entities()
for aid, rec in recs_all.items():
    for role in rec.get("roles", []):
        title = role.get("title", "")
        if title.lower().strip() == "secretary":
            bare_secretary_docs.add(aid)
        elif title.lower().startswith("secretary of") or title.lower().startswith("secretary,"):
            resolved_secretaries[title].add(aid)

print(f"Bare 'Secretary' (unresolved) appears in {len(bare_secretary_docs)} docs")
print(f"Resolved Secretary roles: {len(resolved_secretaries)} distinct titles")
print()
print("Resolved Secretary titles and their doc counts:")
for title, aids in sorted(resolved_secretaries.items(), key=lambda x: -len(x[1])):
    print(f"  {len(aids):3d} docs  {title}")

Bare 'Secretary' (unresolved) appears in 34 docs
Resolved Secretary roles: 38 distinct titles

Resolved Secretary titles and their doc counts:
   84 docs  Secretary of Defense
   51 docs  Secretary of Energy
   51 docs  Secretary of Commerce
   38 docs  Secretary of State
   38 docs  Secretary of Homeland Security
   22 docs  Secretary of Health and Human Services
   21 docs  Secretary of the Treasury
   15 docs  Secretary of Agriculture
   13 docs  Secretary of the Navy
   13 docs  Secretary of Education
   11 docs  Secretary of Transportation
   11 docs  Secretary of Labor
    8 docs  Secretary of the Interior
    6 docs  Secretary of a military department
    6 docs  Secretary of the Air Force
    4 docs  Secretary of Veterans Affairs
    4 docs  Secretary of the Army
    3 docs  Secretary of Housing and Urban Development
    2 docs  Secretary of the Department of Agriculture (resolved via community context and repeated references to Secretary authority over agricultural grants)
   

In [8]:
# What policy domains does each resolved Secretary appear in?
# Shows what information is LOST if you collapse to bare "Secretary"
print("Policy domain spread per resolved Secretary role:")
print("(top application tag per role)\n")

for title, aids in sorted(resolved_secretaries.items(), key=lambda x: -len(x[1]))[:10]:
    tag_counter = collections.Counter()
    for aid in aids:
        for tag in docs_tags.get(aid, []):
            if tag.startswith("Applications:"):
                tag_counter[tag.replace("Applications: ", "")] += 1
    top = tag_counter.most_common(3)
    top_str = ", ".join(f"{t} ({c})" for t, c in top) if top else "no app tag"
    print(f"  {title[:45]:<45}  [{top_str}]")

print()
print("─" * 70)
print(f"If collapsed to bare 'Secretary': 1 node spanning all domains above.")
print(f"Pipeline keeps them separate: {len(resolved_secretaries)} distinct role nodes.")

Policy domain spread per resolved Secretary role:
(top application tag per role)

  Secretary of Defense                           [Government: military and public safety (56), Government: other applications/unspecified (5), Security (5)]
  Secretary of Energy                            [Government: military and public safety (9), Government: other applications/unspecified (8), Energy and utilities (8)]
  Secretary of Commerce                          [Government: military and public safety (12), Manufacturing and process automation (6), Government: other applications/unspecified (6)]
  Secretary of State                             [Government: military and public safety (8), Security (4), Business services and analytics (3)]
  Secretary of Homeland Security                 [Government: military and public safety (11), Government: other applications/unspecified (6), Education (5)]
  Secretary of Health and Human Services         [Medicine, life sciences and public health (4), Security

---
## Demo 4 — Party asymmetry visible only through pipeline entity types

**Setup:** spaCy cannot extract role titles or named policy documents — only person names. The pipeline's `roles` and `named_docs` fields surface entities that carry real partisan signal. This demo computes which entities are exclusive (or near-exclusive) to each party's sponsored bills.

**The question:** What does each party's AI agenda look like when expressed as entity types?

In [9]:
# Build doc -> legislator party mapping
doc_parties = collections.defaultdict(set)   # aid -> set of party chars
leg_party   = {}                              # bioguide -> party

with open(SPONSORS, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        aid   = row.get("AGORA ID", "")
        name  = row.get("Sponsor_Name", "")
        party = row.get("Sponsor_Party", "")
        bg    = row.get("Sponsor_bioguideId", "")
        if name and name != "None" and party in ("D", "R"):
            doc_parties[aid].add(party)
            leg_party[bg] = party

with open(COSPONSOR, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        if row.get("Cosponsor_IsWithdrawn") == "True":
            continue
        aid   = row.get("AGORA ID", "")
        party = row.get("Cosponsor_Party", "")
        bg    = row.get("Cosponsor_BioguideId", "")
        if party in ("D", "R"):
            doc_parties[aid].add(party)
            leg_party[bg] = party

print(f"Docs with party signal: {len(doc_parties)}")

Docs with party signal: 535


In [10]:
# For each entity, count D-docs and R-docs it appears in (binary, one count per doc)
PIPELINE_FIELDS_KEY = {
    "organizations": "name",
    "offices":       "name",
    "roles":         "title",
    "legislation_refs": "name",
    "named_docs":    "name",
}

entity_party_docs = collections.defaultdict(lambda: {"D": set(), "R": set()})

for aid, rec in recs_all.items():
    parties = doc_parties.get(aid, set())
    for field, key in PIPELINE_FIELDS_KEY.items():
        for ent in rec.get(field, []):
            name = ent.get(key, "").strip()
            if not name:
                continue
            label = (name, field)
            for p in parties:
                entity_party_docs[label][p].add(aid)

# Find strongly party-skewed entities (min 10 docs on dominant side, <10% on other)
d_exclusive, r_exclusive = [], []
for (name, field), counts in entity_party_docs.items():
    d = len(counts["D"])
    r = len(counts["R"])
    total = d + r
    if total < 10:
        continue
    if d >= 10 and r / total < 0.1:
        d_exclusive.append((name, field, d, r))
    if r >= 10 and d / total < 0.1:
        r_exclusive.append((name, field, d, r))

d_exclusive.sort(key=lambda x: -x[2])
r_exclusive.sort(key=lambda x: -x[3])

print("DEMOCRAT-SKEWED ENTITIES (≥10 D docs, <10% R)\n")
print(f"  {'D':>4}  {'R':>4}  {'Type':<12}  Entity")
print("  " + "─" * 65)
for name, field, d, r in d_exclusive[:15]:
    print(f"  {d:>4}  {r:>4}  {field[:12]:<12}  {name[:55]}")

print()
print("REPUBLICAN-SKEWED ENTITIES (≥10 R docs, <10% D)\n")
print(f"  {'D':>4}  {'R':>4}  {'Type':<12}  Entity")
print("  " + "─" * 65)
for name, field, d, r in r_exclusive[:15]:
    print(f"  {d:>4}  {r:>4}  {field[:12]:<12}  {name[:55]}")

DEMOCRAT-SKEWED ENTITIES (≥10 D docs, <10% R)

     D     R  Type          Entity
  ─────────────────────────────────────────────────────────────────

REPUBLICAN-SKEWED ENTITIES (≥10 R docs, <10% D)

     D     R  Type          Entity
  ─────────────────────────────────────────────────────────────────


In [11]:
# Spotlight: "Developer" and "Deployer" role titles — purely Democratic AI governance concepts
for target in ["Developer", "Deployer"]:
    counts = entity_party_docs.get((target, "roles"), {"D": set(), "R": set()})
    d, r = len(counts["D"]), len(counts["R"])
    print(f'Role title "{target}": D={d} docs  R={r} docs')
    # Show one example context
    for aid, rec in recs_all.items():
        for role in rec.get("roles", []):
            if role.get("title","").strip() == target:
                print(f'  example context: "{role.get("context","")[:100]}"')
                break
        else:
            continue
        break
    print()

Role title "Developer": D=3 docs  R=1 docs
  example context: "The developer of a critical-impact artificial intelligence system that agrees through a contract or "

Role title "Deployer": D=1 docs  R=0 docs
  example context: "provide deployers of the covered algorithm a mechanism to comply with subsection"



---
## Demo 5 — Entity graph neighborhood: one bill as a structured object

**Setup:** The pipeline produces a knowledge graph (`layer_3_entity.graphml`) where document nodes connect to every entity they reference. This demo loads the subgraph around the highest-degree document and shows what it looks like as a structured object vs. a raw text blob.

**The question:** What does document:77 — the hub with the most entity connections (230 edges) — look like as a graph neighborhood?

In [12]:
import networkx as nx

G = nx.read_graphml(str(GRAPH))
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Find hub documents by degree
U = G.to_undirected()
doc_nodes = [(n, U.degree(n), G.nodes[n].get("label", n))
             for n, d in G.nodes(data=True) if d.get("node_type") == "Document"]
doc_nodes.sort(key=lambda x: -x[1])

print("\nTop 10 hub documents:")
print(f"  {'Degree':>6}  Node ID")
for nid, deg, label in doc_nodes[:10]:
    print(f"  {deg:>6}  {label}")

Graph: 5366 nodes, 10943 edges

Top 10 hub documents:
  Degree  Node ID
     230  document:77
     182  document:75
     147  document:457
     136  document:1214
     134  document:2367
     131  document:1201
     125  document:25
     114  document:845
     113  document:1690
     109  document:860


In [13]:
HUB_DOC = "document:77"   # change to any doc node id from the list above

neighbors = list(U.neighbors(HUB_DOC))
by_type = collections.defaultdict(list)
for nb in neighbors:
    ntype = G.nodes[nb].get("node_type", "?")
    label = G.nodes[nb].get("label", nb)
    deg   = U.degree(nb)
    by_type[ntype].append((label, deg))

agora_id = HUB_DOC.replace("document:", "")
print(f"Neighborhood of {HUB_DOC}  (agora_id={agora_id})")
print(f"Total connected entities: {len(neighbors)}\n")

for ntype in ["Org", "Role", "Office", "Legislation", "Named Doc"]:
    items = sorted(by_type.get(ntype, []), key=lambda x: -x[1])
    if not items:
        continue
    print(f"{ntype.upper()} ({len(items)})")
    for label, deg in items[:12]:
        print(f"  {'·'} {label[:70]}")
    if len(items) > 12:
        print(f"    … {len(items)-12} more")
    print()

Neighborhood of document:77  (agora_id=77)
Total connected entities: 228

ORG (90)
  · Department of Defense
  · Department of Energy
  · Department of Commerce
  · National Institute of Standards and Technology
  · Congress
  · President
  · National Science Foundation
  · Government Accountability Office
  · People's Republic of China
  · National Laboratories
  · Department of Health and Human Services
  · United States
    … 78 more

ROLE (24)
  · Director
  · Secretary of Commerce
  · Secretary of Energy
  · Comptroller General of the United States
  · Director of National Intelligence
  · Director of the National Science Foundation
  · Secretary of Education
  · Director of the Office of Science and Technology Policy
  · Director of the Office of Management and Budget
  · Director of the Office of Personnel Management
  · head of any Federal agency
  · Director of the Federal Bureau of Investigation
    … 12 more

OFFICE (21)
  · Director of the National Institute of Standards an

In [14]:
# Which of those entity nodes are also connected to many OTHER documents?
# Those are the "universal" entities — the cross-cutting nodes of the graph.
print("Cross-cutting entities in this hub's neighborhood")
print("(connected to this doc AND ≥10 other docs)\n")
print(f"  {'This-doc deg':>12}  {'Graph deg':>9}  {'Type':<12}  Entity")
print("  " + "─" * 65)

cross_cutting = []
for nb in neighbors:
    ntype = G.nodes[nb].get("node_type", "?")
    if ntype == "Document":
        continue
    label   = G.nodes[nb].get("label", nb)
    g_deg   = U.degree(nb)
    # count how many document neighbors this entity has
    doc_neighbors = sum(1 for n in U.neighbors(nb) if G.nodes[n].get("node_type") == "Document")
    if doc_neighbors >= 10:
        cross_cutting.append((label, ntype, g_deg, doc_neighbors))

cross_cutting.sort(key=lambda x: -x[3])
for label, ntype, g_deg, doc_cnt in cross_cutting[:20]:
    print(f"  {doc_cnt:>12} docs  {g_deg:>9}  {ntype:<12}  {label[:50]}")

print(f"\n{len(cross_cutting)} cross-cutting entities in this neighborhood out of {len(neighbors)} total.")

Cross-cutting entities in this hub's neighborhood
(connected to this doc AND ≥10 other docs)

  This-doc deg  Graph deg  Type          Entity
  ─────────────────────────────────────────────────────────────────
            69 docs         76  Role          Director
            64 docs         73  Org           Department of Defense
            59 docs         65  Org           Department of Energy
            57 docs         62  Org           Department of Commerce
            55 docs         61  Org           National Institute of Standards and Technology
            52 docs         57  Role          Secretary of Commerce
            49 docs         52  Role          Secretary of Energy
            48 docs         56  Org           Congress
            47 docs         56  Legislation   National Artificial Intelligence Initiative Act of
            45 docs         49  Legislation   William M. (Mac) Thornberry National Defense Autho
            43 docs         48  Legislation   Higher Ed

---

## What these demos collectively show

| Demo | What a keyword/regex approach gives you | What the pipeline gives you |
|---|---|---|
| 1 | The string `"secretary"` — 88 hits | 41 distinct cabinet officials, each pinned to a bill section |
| 2 | A list of matching spans | Structured entities with type, acronym, parent org, and extractive context |
| 3 | One overloaded "Secretary" graph node | Separate nodes for DoD, HHS, Agriculture, Energy… each in its own policy domain |
| 4 | Bag-of-words party comparison | Role titles ("Developer", "Deployer") and legislation as partisan signals |
| 5 | A text file | A graph: 230 entity connections, typed, traversable, cross-document |

The common thread: the pipeline produces **relational structure**, not just occurrence lists. The graph edges are the output — they encode which entities co-appear in the same legislative instrument, which is the unit of analysis for AI policy research.

> **Interactive visualization:** open `reports/community_001_network.html` in a browser for a clickable version of the military community subgraph (294 nodes, 1001 edges).